In [0]:
SHOW VOLUMES IN dbw_taxrisk_dev.bronze;

Country profile table

In [0]:
CREATE TABLE IF NOT EXISTS dbw_taxrisk_dev.config.country_profile (
  country_code       STRING NOT NULL COMMENT 'ISO 3166-1 alpha-2, e.g. CO',
  country_name_en    STRING,
  country_name_es    STRING,
  languages          ARRAY<STRING> COMMENT 'ISO 639-1 codes',
  currency_code      STRING COMMENT 'ISO 4217, e.g. COP',
  tax_authority      STRING,
  privacy_regime     STRING,
  disclosure_lexicon ARRAY<STRING> COMMENT 'Seed terms that signal sponsorship or exchange',
  enabled_sources    ARRAY<STRING> COMMENT 'Data sources enabled for this country',
  is_oecd            BOOLEAN,
  is_ciat            BOOLEAN,
  CONSTRAINT pk_country_profile PRIMARY KEY (country_code)
)
COMMENT 'One row per country. Selected by the country_code job parameter.';

Tax rules table

In [0]:
CREATE TABLE IF NOT EXISTS dbw_taxrisk_dev.config.country_tax_rules (
  country_code        STRING NOT NULL,
  valid_from          DATE NOT NULL COMMENT 'Rule applies from this date',
  valid_to            DATE COMMENT 'NULL means still in force',
  in_kind_treatment   STRING COMMENT 'taxable | conditional | not_taxable | unknown',
  assumed_income_rate DECIMAL(5,4) COMMENT 'Flat rate used only for exposure estimates',
  indirect_tax_note   STRING,
  filing_threshold    DECIMAL(18,2) COMMENT 'Local currency; NULL if none or unknown',
  source_reference    STRING COMMENT 'Where the rule comes from',
  is_placeholder      BOOLEAN COMMENT 'TRUE until a tax professional validates the row',
  CONSTRAINT pk_country_tax_rules PRIMARY KEY (country_code, valid_from)
)
COMMENT 'Versioned tax rules per country.';

Seed Colombia

In [0]:
DELETE FROM dbw_taxrisk_dev.config.country_profile WHERE country_code = 'CO';
INSERT INTO dbw_taxrisk_dev.config.country_profile VALUES (
  'CO', 'Colombia', 'Colombia', array('es'), 'COP', 'DIAN', 'Ley 1581 de 2012',
  array('#canje', '#publicidad', '#colaboración', '#regalo'), array('synthetic'), true, true);

DELETE FROM dbw_taxrisk_dev.config.country_tax_rules WHERE country_code = 'CO';
INSERT INTO dbw_taxrisk_dev.config.country_tax_rules VALUES (
  'CO', DATE'2024-01-01', NULL, 'taxable', 0.2000, 'Not assessed', NULL,
  'PLACEHOLDER: validate with a Colombian tax professional', true);

Validation

In [0]:
SELECT * FROM dbw_taxrisk_dev.config.country_profile;

In [0]:
SELECT * FROM dbw_taxrisk_dev.config.country_tax_rules;